In [1]:
import os
import joblib
import imageio
import numpy as np
import pandas as pd
import netCDF4 as nc
import seaborn as sns
import matplotlib.pyplot as plt

# PATHS AND FOLDERS
measurementpath = 'S:/pools/t/T-IDP-Projekte-u-Vorlesungen/Meteoblue/Data/Messdaten/Daten_Meteoblue'

faulty_stations = ['C059A2225266', 'D63DFE9B164B', 'D07769DF208C', 'DF15D23E4B15', 'E2A0DF1A4941', 'E437CB2AF225', 'F5C16A4B6340',
                   'F033A8C6BB79', 'F4683D808CFB', 'D3FE8EEF188C', 'D883D89E6A24', 'EC032D8260EB', 'C3FD36A6C1BC', 'D083B9FD07FB', 
                   'EB90524D4F3E', 'FCBBD3B1DB2C']

palm_paths = {'mb4': 'mb_4_multi_stations_xy_N02.00m.nc',
              'mb5': 'mb_5_multi_stations_LCZ_xy_N02.00m.nc',
              'mb6': 'mb_6_multi_stations_LCZ_xy_N02.00m.nc',
              'mb7': 'mb_7_multi_stations_LCZ_xy_N02.00m.nc',
              'mb8': 'mb_8_multi_stations_LCZ_xy_N02.00m.nc'}

def load_file(datapath):
    with open(datapath, 'rb') as file:
        data = joblib.load(file)
        file.close()
    return data

# GAN Training Dataset
The GAN requires low- and high-resolution images as input. The high-resolution images are generated from the results of the QRF inference runs, done on feature maps from a PALM simulation time window. The low-resolution counterparts are empty (NaN) arrays with measurements at station locations. 

Settings:

In [2]:
palmref = 'mb5'
srfactor = 10
savefolder = f'S:/pools/t/T-IDP-Projekte-u-Vorlesungen/Meteoblue/QRF/Data/GAN Maps/mb5_{srfactor}x'
runname = '2024-3-5_15.12'
inferencedatapath = f'S:/pools/t/T-IDP-Projekte-u-Vorlesungen/Meteoblue/QRF/Data/QRF Inference/{runname}/{runname}.json'
inferencedata = load_file(inferencedatapath)
temps = inferencedata[:, :, :, 1]

## Low-Resolution Temperature Maps
Functions:

In [3]:
def palm_times(palmfile: nc.Dataset):
    """
    Extracts the time vector, formatting it as a datetime. The time contained within the PALM file is given as
    minutes since origin. Additionally, a boolean vector is generated, indicating the start of the useable time
    series (certain observations are required to create the moving average).
    """
    origintime = pd.to_datetime(palmfile.origin_time)
    times_list = palmfile['time']
    times = []
    for _, time in enumerate(times_list):
        times.append(origintime + pd.Timedelta(minutes=np.round(time * 24 * 60)))
    if not times:
        raise ValueError
    return times


def empty_array(palmdim):
    """
    Generates an empty array of the same dimensions as the PALM file.
    """
    empty_array = np.empty((palmdim[0]-2, palmdim[2], palmdim[3]), dtype=np.float32)
    empty_array[:] = np.nan
    return empty_array


def lv03_to_lv95(lv03_lat: float, lv03_lon: float):
    return lv03_lat + 1000000, lv03_lon + 2000000


def coordinates(palmfile, res=16):
    CH_S, CH_W = lv03_to_lv95(palmfile.origin_y, palmfile.origin_x)
    # CH_S, CH_W, _ = wgs84_to_lv(palmfile.origin_lat, palmfile.origin_lon, 'lv95') #type: ignore
    CH_N = CH_S + palmfile.dimensions['y'].size * res
    CH_E = CH_W + palmfile.dimensions['x'].size * res
    return CH_N, CH_E, CH_S, CH_W


def palm_info(palmfile):
    info = {}
    info['CH_N'], info['CH_E'], info['CH_S'], info['CH_W'] = coordinates(palmfile)
    return info


def stations_loc(boundary):
    stationscsv = pd.read_csv('S:/pools/t/T-IDP-Projekte-u-Vorlesungen/Meteoblue/Data/Messdaten/stations_new.csv', delimiter=';')
    stationsloc = {}
    stations = stationscsv['stationid_new'].unique()
    for station in stations:
        row = stationscsv[stationscsv['stationid_new'] == station]
        if not row.empty:
            if boundary['CH_W'] <= int(row["CH_E"]) <= boundary['CH_E'] and /
               boundary['CH_S'] <= int(row["CH_N"]) <= boundary['CH_N']:
                stationsloc[station] = {'lat': int(row["CH_N"]), 'lon': int(row["CH_E"]), 
                                        'lat_idx': int((boundary['CH_N'] - row['CH_N']) / 16), 
                                        'lon_idx': int((row['CH_E'] - boundary['CH_W']) / 16)}
    return stationsloc


def extract_measurements(measurementpath, stationid, times):
    try:
        measurementfile= pd.read_csv(f'{measurementpath}/temp_{stationid}.csv', delimiter=';')
    except FileNotFoundError:
        return [np.nan] * len(times)
    
    true_temps = []
    measurementfile['datetime_round'] = pd.to_datetime(measurementfile['datetime']).dt.round('30min')
    times = pd.to_datetime(times)
    for t in times:
        t = t.tz_localize(None)
        try:
            temp = np.mean(measurementfile[measurementfile['datetime_round'] == t]['temp'])
            true_temps.append(temp)
        except IndexError as e:
            print(measurementfile['datetime'])
            print(t)
            raise e
                
    return true_temps


def fill_maps(mapshape, stationsloc, measurementpath, times):
    temparray = empty_array(mapshape)
    for station in stationsloc.keys():
        temps = extract_measurements(measurementpath, station, times)
        if not np.isnan(temps).all():
            lat_idx = stationsloc[station]['lat_idx']
            lon_idx = stationsloc[station]['lon_idx']
            temparray[:, lat_idx, lon_idx] = temps
    return temparray


def lower_resolution(hr_map, sr_factor):
    lr_map = np.zeros((hr_map.shape[0], hr_map.shape[1] // sr_factor, hr_map.shape[2] // sr_factor))
    for i in range(lr_map.shape[1]):
        for j in range(lr_map.shape[2]):
            for t in range(lr_map.shape[0]):
                lr_map[t, i, j] = np.nanmean(hr_map[t, i*sr_factor : (i+1)*sr_factor, 
                                                       j*sr_factor : (j+1)*sr_factor])
    return lr_map


def highres_maps(palmref, measurementpath):
    """
    Generates a low resolution map of the PALM file, using measurements from the stations within the boundary.
    """
    palmfile = nc.Dataset(f'S:/pools/t/T-IDP-Projekte-u-Vorlesungen/Meteoblue/Data/PALM Maps/{palm_paths[palmref]}', 'r')
    boundary = palm_info(palmfile)
    datetimes = palm_times(palmfile)[2:]
    stationsloc = stations_loc(boundary)
    hr_filled_map = fill_maps(palmfile['theta_xy'].shape, stationsloc, measurementpath, datetimes)
    return hr_filled_map, datetimes

In [4]:
highresmap, datetimes = highres_maps(palmref, measurementpath)
lowresmap = lower_resolution(highresmap, srfactor)
datetimes = datetimes[2:]

C:\Users\ushe\AppData\Local\Temp\ipykernel_24084\2748487654.py:51: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  if boundary['CH_W'] <= int(row["CH_E"]) <= boundary['CH_E'] and \
C:\Users\ushe\AppData\Local\Temp\ipykernel_24084\2748487654.py:52: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  boundary['CH_S'] <= int(row["CH_N"]) <= boundary['CH_N']:
C:\Users\ushe\AppData\Local\Temp\ipykernel_24084\2748487654.py:53: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  stationsloc[station] = {'lat': int(row["CH_N"]), 'lon': int(row["CH_E"]),
C:\Users\ushe\AppData\Local\Temp\ipykernel_24084\2748487654.py:54: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. U

In [5]:
# Set min and max values for images
tmin = np.nanmin([np.nanmin(highresmap), np.nanmin(temps)])
tmax = np.nanmax([np.nanmax(highresmap), np.nanmax(temps)])

Scaled images (for viewing)

In [6]:
imgfolder = f'{savefolder}/images'
if not os.path.exists(imgfolder):
    os.makedirs(imgfolder)

for idx, time in enumerate(datetimes):
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    ax = sns.heatmap(lowresmap[idx, :, :], xticklabels=False, yticklabels=False, vmin=tmin, vmax=tmax, cmap='viridis')
    ax.set_title(f'LR image at time {time}')
    plt.savefig(f'{imgfolder}/{idx}_lr.png', bbox_inches='tight', pad_inches=0)
    plt.close(fig)

GAN training images

In [7]:
imgfolder = f'{savefolder}/training'
if not os.path.exists(imgfolder):
    os.makedirs(imgfolder)

tmax = np.nanmax(lowresmap)
tmin = np.nanmin(lowresmap)
for idx, time in enumerate(datetimes):
    plt.imsave(f'{imgfolder}/{idx}_lr.png', lowresmap[idx, :, :], cmap='viridis', vmin=tmin, vmax=tmax)

## High-Resolution Temperature Maps
The high-resolution counterparts are extracted from the QRF prediction maps, using the mean value and disregarding the quantile predictions.

Scaled images (for viewing)

In [8]:
imgfolder = f'{savefolder}/images'
if not os.path.exists(imgfolder):
    os.makedirs(imgfolder)

for idx, time in enumerate(datetimes):
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    ax = sns.heatmap(temps[idx, :, :], xticklabels=False, yticklabels=False, vmin=tmin, vmax=tmax, cmap='viridis')
    ax.set_title(f'LR image at time {time}')
    plt.savefig(f'{imgfolder}/{idx}_hr.png', bbox_inches='tight', pad_inches=0)
    plt.close(fig)

GAN training images

In [9]:
imgfolder = f'{savefolder}/training'
if not os.path.exists(imgfolder):
    os.makedirs(imgfolder)

for idx, time in enumerate(datetimes):
    plt.imsave(f'{imgfolder}/{idx}_hr.png', temps[idx, :, :], cmap='viridis', vmin=tmin, vmax=tmax)

## Alternative: HR Maps from PALM


In [ ]:
import netCDF4 as nc

palmfile = f'S:/pools/t/T-IDP-Projekte-u-Vorlesungen/Meteoblue/Data/PALM Maps{palm_paths[palmref]}'


In [11]:
a = ['.nc', '.nc'] 
b = ['.json', '.nc']

set(a).intersection(b)

{'.nc'}